In [19]:
import os
import time
from langchain_groq import ChatGroq  
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


os.environ["GROQ_API_KEY"] = "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"

print("Initializing Groq model...")

llm = ChatGroq(
    model="llama-3.3-70b-versatile", # Using Meta's Llama 3 model hosted on Groq
    temperature=0,
    max_retries=3 
)

# ==========================================
# 1. Basic LLM Call
# ==========================================
print("\n--- 1. Basic LLM Call ---")
basic_response = llm.invoke("What is 2 + 2?")
print(basic_response.content)

time.sleep(2) # Tiny pause for rate limits


# ==========================================
# 2. PromptTemplate & 3. Simple Chain
# ==========================================
print("\n--- 2 & 3. Simple Chain with Template ---")
joke_template = PromptTemplate.from_template("Tell me a short technical joke about {topic}.")
chain = joke_template | llm | StrOutputParser()

chain_response = chain.invoke({"topic": "Python developers"})
print(chain_response)

time.sleep(2)


# ==========================================
# 4. Agent Tool Calling (Pure LCEL Approach)
# ==========================================
print("\n--- 4. Tool Calling: Under the Hood ---")

@tool
def get_system_time(format_type: str = "standard") -> str:
    """Returns the current system time. Always use this tool when asked about the time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [get_system_time]
# LangChain makes this universal! .bind_tools works exactly the same on Groq as it did on Gemini.
llm_with_tools = llm.bind_tools(tools) 

user_query = "Can you check the system time for me?"
print(f"User: {user_query}")

response = llm_with_tools.invoke(user_query)

if response.tool_calls:
    print("\n[System] The LLM chose to use a tool instead of generating text.")
    for tool_call in response.tool_calls:
        print(f"[System] Tool Requested: {tool_call['name']}")
        print(f"[System] Arguments passed by LLM: {tool_call['args']}")
        
        if tool_call['name'] == "get_system_time":
            tool_result = get_system_time.invoke(tool_call['args'])
            print(f"[System] Tool Output Generated: {tool_result}")
else:
    print("Agent:", response.content)

time.sleep(2)


# ==========================================
# 5. Memory Example (Modern LCEL Approach)
# ==========================================
print("\n--- 5. Modern Memory Example ---")

memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

memory_chain = memory_prompt | llm | StrOutputParser()
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    memory_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

config = {"configurable": {"session_id": "test_session_001"}}

# Interaction 1
print("User: Hi, my name is Alex and I am learning GenAI.")
res1 = chain_with_memory.invoke({"input": "Hi, my name is Alex and I am learning GenAI."}, config=config)
print("Agent:", res1)

time.sleep(2) 

# Interaction 2
print("\nUser: What is my name and what am I learning?")
res2 = chain_with_memory.invoke({"input": "What is my name and what am I learning?"}, config=config)
print("Agent:", res2)

print("\n🎉 SUCCESS! You just ran a LangChain pipeline using Groq and Llama 3!")

Initializing Groq model...

--- 1. Basic LLM Call ---
2 + 2 = 4.

--- 2 & 3. Simple Chain with Template ---
Why do Python developers prefer dark mode. Because light attracts bugs.

--- 4. Tool Calling: Under the Hood ---
User: Can you check the system time for me?

[System] The LLM chose to use a tool instead of generating text.
[System] Tool Requested: get_system_time
[System] Arguments passed by LLM: {'format_type': 'standard'}
[System] Tool Output Generated: 2026-04-16 22:58:56

--- 5. Modern Memory Example ---
User: Hi, my name is Alex and I am learning GenAI.
Agent: Hello Alex, nice to meet you. That's great that you're learning about GenAI (General Artificial Intelligence). It's an exciting and rapidly evolving field. What specific aspects of GenAI are you interested in or would you like to learn more about? I'm here to help and provide guidance as you explore this fascinating topic.

User: What is my name and what am I learning?
Agent: Your name is Alex, and you are learning abo